In [30]:
import math
import torch
import torch.nn as nn

In [ ]:
def autopad(k, p=None, d=1):
    """Pad to 'same'. k=kernel, p=pad, d=dilation."""
    if d > 1:
        k = d * (k - 1) + 1
    if p is None:
        p = k // 2
    return p


class Conv(nn.Module):
    """Standard Conv2d + BatchNorm + SiLU  (the ultralytics 'Conv')."""
    default_act = nn.SiLU()

    def __init__(self, c1, c2, k=1, s=1, p=None, g=1, d=1, act=True):
        super().__init__()
        self.conv = nn.Conv2d(c1, c2, k, s, autopad(k, p, d), groups=g,
                              dilation=d, bias=False)
        self.bn = nn.BatchNorm2d(c2)
        self.act = self.default_act if act is True else (
            act if isinstance(act, nn.Module) else nn.Identity())

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))


class DWConv(Conv):
    """Depthwise Conv (groups = gcd(c1, c2))."""
    def __init__(self, c1, c2, k=1, s=1, d=1, act=True):
        super().__init__(c1, c2, k, s, g=math.gcd(c1, c2), d=d, act=act)


class GhostConv(nn.Module):
    """
    GhostConv (GhostNet, 2020) — the backbone STEM.
    Paper §2.1: applied ONLY at the input stage; 'cheap' ops reconstruct
    half the channels instead of full convolutions.
    """
    def __init__(self, c1, c2, k=1, s=1, g=1, act=True):
        super().__init__()
        c_ = c2 // 2
        self.primary = Conv(c1, c_, k, s, None, g, act=act)
        self.cheap = Conv(c_, c_, 5, 1, None, g=c_, act=act)  

    def forward(self, x):
        y = self.primary(x)
        return torch.cat((y, self.cheap(y)), 1)

In [32]:
class LayerNorm2d(nn.Module):
    """LayerNorm over the channel dim of an (N, C, H, W) tensor."""
    def __init__(self, c, eps=1e-6):
        super().__init__()
        self.norm = nn.LayerNorm(c, eps=eps)

    def forward(self, x):
        x = x.permute(0, 2, 3, 1)
        x = self.norm(x)
        return x.permute(0, 3, 1, 2).contiguous()


class GatedCNNBlock(nn.Module):
    """
    Faithful MambaOut Gated CNN block (Yu & Wang, 2025) — what Fig 3c depicts:
    "gated fusion between two linear paths and a convolution branch".
    Paper Eqs 1-2 are compressed notation for it.
        x̂       = LayerNorm(x)
        g, i, c = split(fc1(x̂))                  # gate | identity | conv paths
        c       = DWConv(c)                       # convolution branch
        x       = fc2( act(g) ⊙ concat(i, c) )    # gated fusion + w3 proj
        out     = x + shortcut
    expansion_ratio / conv_ratio use MambaOut defaults; these are the first
    knobs to turn if the full-model param count misses 12.05M.
    """
    def __init__(self, dim, expansion_ratio=8/3, kernel_size=7,
                 conv_ratio=1.0, act_layer=nn.GELU):
        super().__init__()
        self.norm = LayerNorm2d(dim)
        hidden = int(expansion_ratio * dim)
        self.fc1 = nn.Conv2d(dim, hidden * 2, 1)
        self.act = act_layer()
        conv_ch = int(conv_ratio * dim)
        self.split = (hidden, hidden - conv_ch, conv_ch)
        self.conv = nn.Conv2d(conv_ch, conv_ch, kernel_size,
                              padding=kernel_size // 2, groups=conv_ch)
        self.fc2 = nn.Conv2d(hidden, dim, 1)   # w3 output projection

    def forward(self, x):
        shortcut = x
        x = self.norm(x)
        g, i, c = torch.split(self.fc1(x), self.split, dim=1)
        c = self.conv(c)
        x = self.fc2(self.act(g) * torch.cat((i, c), dim=1))
        return x + shortcut

In [33]:
class DMambaOut(nn.Module):
    """
    Dual-path (CSP-style) wrapper around Gated CNN blocks (Paper Fig 3b, Eqs 3-6):
        x1, x2 = Split(Conv1x1(x))     # x1 = shortcut, x2 = modeling seed
        y_i    = f_i(y_{i-1})          # chain of n Gated CNN blocks
        y      = Conv1x1( Concat(x1, y_1..y_n) )
    Ambiguity: Eq 4 literally seeds with x1 (leaving x2 unused) — reads as a
    typo. Sensible CSP topology is default; literal_seed=True reproduces the text.
    """
    def __init__(self, c, n=1, literal_seed=False):
        super().__init__()
        self.c_ = c // 2
        self.literal_seed = literal_seed
        self.cv1 = Conv(c, 2 * self.c_, 1, 1)
        self.blocks = nn.ModuleList(GatedCNNBlock(self.c_) for _ in range(n))
        self.cv2 = Conv((1 + n) * self.c_, c, 1, 1)

    def forward(self, x):
        x1, x2 = self.cv1(x).split((self.c_, self.c_), dim=1)
        seed = x1 if self.literal_seed else x2
        outs, y = [], seed
        for blk in self.blocks:
            y = blk(y)
            outs.append(y)
        return self.cv2(torch.cat([x1, *outs], dim=1))


class GMOBlock(nn.Module):
    """One downsampling Conv + k stacked DMambaOut modules (Paper §2.1)."""
    def __init__(self, c1, c2, k=1, stride=2, inner_n=1):
        super().__init__()
        self.down = Conv(c1, c2, 3, stride)
        self.aggr = nn.Sequential(*[DMambaOut(c2, n=inner_n) for _ in range(k)])

    def forward(self, x):
        return self.aggr(self.down(x))

In [34]:
class GMONet(nn.Module):
    """
    GhostConv stem (s2) + GMO-Block1..4.
    Paper channels: 128, 256, 384, 384   |   depths: 1, 1, 1, 3
    Neck taps the last three stages -> strides 8 / 16 / 32.
    """
    def __init__(self, in_ch=3, stem_ch=64,
                 widths=(128, 256, 384, 384),
                 depths=(1, 1, 1, 3)):
        super().__init__()
        self.stem = GhostConv(in_ch, stem_ch, k=3, s=2)
        c_prev = stem_ch
        self.stages = nn.ModuleList()
        for w, d in zip(widths, depths):
            self.stages.append(GMOBlock(c_prev, w, k=d, stride=2))
            c_prev = w
        self.out_channels = list(widths[1:])   # channels of the 3 tapped maps

    def forward(self, x):
        x = self.stem(x)
        feats = []
        for st in self.stages:
            x = st(x)
            feats.append(x)
        return feats[1], feats[2], feats[3]     # S3, S4, S5

In [35]:
torch.manual_seed(0)
net = GMONet()
x = torch.randn(1, 3, 640, 640)
s3, s4, s5 = net(x)

n_params = sum(p.numel() for p in net.parameters())
print("input           :", tuple(x.shape))
print("S3 (stride  8)  :", tuple(s3.shape))
print("S4 (stride 16)  :", tuple(s4.shape))
print("S5 (stride 32)  :", tuple(s5.shape))
print(f"backbone params : {n_params/1e6:.2f} M")

assert s3.shape[1] == 256 and s3.shape[2] == 80, "S3 wrong (want 256ch, 80x80)"
assert s4.shape[1] == 384 and s4.shape[2] == 40, "S4 wrong (want 384ch, 40x40)"
assert s5.shape[1] == 384 and s5.shape[2] == 20, "S5 wrong (want 384ch, 20x20)"

loss = sum(f.mean() for f in (s3, s4, s5))
loss.backward()
assert all(p.grad is not None for p in net.parameters() if p.requires_grad), \
    "some params received no gradient"

print("\nOK — GMONet emits correct feature maps and is fully differentiable.")

input           : (1, 3, 640, 640)
S3 (stride  8)  : (1, 256, 80, 80)
S4 (stride 16)  : (1, 384, 40, 40)
S5 (stride 32)  : (1, 384, 20, 20)
backbone params : 5.33 M

OK — GMONet emits correct feature maps and is fully differentiable.


In [36]:
class GSConv(nn.Module):
    """Slim-neck GSConv (Li et al. 2024). Eager: parser prepends c1, YAML gives c2(+k,s)."""
    def __init__(self, c1, c2, k=1, s=1, g=1, act=True):
        super().__init__()
        c_ = c2 // 2
        self.cv1 = Conv(c1, c_, k, s, g=1, act=act)
        self.cv2 = Conv(c_, c_, 5, 1, g=c_, act=act)      # depthwise DSConv

    def forward(self, x):
        x1 = self.cv1(x)
        y = torch.cat((x1, self.cv2(x1)), 1)
        b, n, h, w = y.size()
        y = y.view(b, 2, n // 2, h, w).transpose(1, 2).contiguous().view(b, n, h, w)
        return y

In [37]:
class RepConv(nn.Module):
    """Reparameterizable conv (3x3 + 1x1 branches) with fuse_convs() so
    Ultralytics' model.fuse() works during final validation and inference."""
    default_act = nn.SiLU()

    def __init__(self, c1, c2, k=3, s=1, p=1, g=1, act=True):
        super().__init__()
        self.g, self.c1, self.c2 = g, c1, c2
        self.act = self.default_act if act is True else (
            act if isinstance(act, nn.Module) else nn.Identity())
        self.conv1 = Conv(c1, c2, k, s, p=p, g=g, act=False)
        self.conv2 = Conv(c1, c2, 1, s, p=(p - k // 2), g=g, act=False)

    def forward(self, x):
        return self.act(self.conv1(x) + self.conv2(x))

    def forward_fuse(self, x):
        return self.act(self.conv(x))

    def _pad_1x1_to_3x3(self, w):
        return None if w is None else torch.nn.functional.pad(w, [1, 1, 1, 1])

    def _fuse_bn(self, branch):
        if branch is None:
            return 0, 0
        conv, bn = branch.conv, branch.bn
        std = (bn.running_var + bn.eps).sqrt()
        t = (bn.weight / std).reshape(-1, 1, 1, 1)
        return conv.weight * t, bn.bias - bn.running_mean * bn.weight / std

    def get_equivalent_kernel_bias(self):
        k1, b1 = self._fuse_bn(self.conv1)
        k2, b2 = self._fuse_bn(self.conv2)
        return k1 + self._pad_1x1_to_3x3(k2), b1 + b2

    def fuse_convs(self):
        if hasattr(self, "conv"):
            return
        kernel, bias = self.get_equivalent_kernel_bias()
        self.conv = nn.Conv2d(self.conv1.conv.in_channels, self.conv1.conv.out_channels,
                              kernel_size=3, stride=self.conv1.conv.stride,
                              padding=self.conv1.conv.padding, groups=self.g,
                              bias=True).requires_grad_(False)
        self.conv.weight.data = kernel
        self.conv.bias.data = bias
        for p in self.parameters():
            p.detach_()
        self.__delattr__("conv1")
        self.__delattr__("conv2")


class RepNBottleneck(nn.Module):
    def __init__(self, c1, c2, shortcut=True, g=1, e=0.5):
        super().__init__()
        c_ = int(c2 * e)
        self.cv1 = RepConv(c1, c_, 3, 1)
        self.cv2 = Conv(c_, c2, 3, 1, g=g)
        self.add = shortcut and c1 == c2

    def forward(self, x):
        return x + self.cv2(self.cv1(x)) if self.add else self.cv2(self.cv1(x))


class RepNCSP(nn.Module):
    def __init__(self, c1, c2, n=1, shortcut=True, g=1, e=0.5):
        super().__init__()
        c_ = int(c2 * e)
        self.cv1 = Conv(c1, c_, 1, 1)
        self.cv2 = Conv(c1, c_, 1, 1)
        self.cv3 = Conv(2 * c_, c2, 1)
        self.m = nn.Sequential(*(RepNBottleneck(c_, c_, shortcut, g, e=1.0)
                                 for _ in range(n)))

    def forward(self, x):
        return self.cv3(torch.cat((self.m(self.cv1(x)), self.cv2(x)), 1))

In [38]:
class CAA(nn.Module):
    """
    Context Anchor Attention — PKINet-faithful (Cai et al. 2024, Eq 5 / Fig 2e).
    AvgPool -> 1x1 (reduce to C/r) -> horizontal DW (1xk) -> vertical DW (kx1)
    -> 1x1 (expand to C) -> sigmoid -> multiply. Strip convs run in the reduced
    space (r), matching PKINet; kernel length h_k/v_k default 11 (paper doesn't
    print exact values, so tunable).
    """
    def __init__(self, ch, reduction=4, h_kernel=11, v_kernel=11):
        super().__init__()
        rc = max(ch // reduction, 8)
        self.avg_pool = nn.AvgPool2d(7, 1, 3)
        self.conv1 = Conv(ch, rc, 1)                                  # reduce
        self.h_conv = nn.Conv2d(rc, rc, (1, h_kernel), 1,
                                (0, h_kernel // 2), groups=rc)        # horizontal strip
        self.v_conv = nn.Conv2d(rc, rc, (v_kernel, 1), 1,
                                (v_kernel // 2, 0), groups=rc)        # vertical strip
        self.conv2 = Conv(rc, ch, 1)                                  # expand
        self.act = nn.Sigmoid()

    def forward(self, x):
        attn = self.conv1(self.avg_pool(x))
        attn = self.h_conv(attn)
        attn = self.v_conv(attn)
        attn = self.act(self.conv2(attn))
        return x * attn

In [39]:
class CAFF(nn.Module):
    """Context-Aware Feature Fusion — replaces RepC3. Eager: parser prepends c1."""
    def __init__(self, c1, c2, c3=None, c4=None, n=1):
        super().__init__()
        c3 = c3 or c2
        c4 = c4 or c2 // 2
        self.cv1 = Conv(c1, c3, 1, 1)
        self.cv2 = nn.Sequential(RepNCSP(c3 // 2, c4, n), Conv(c4, c4, 3, 1))
        self.cv3 = nn.Sequential(RepNCSP(c4, c4, n), Conv(c4, c4, 3, 1))
        self.caa = CAA(c3 + 2 * c4)
        self.cv4 = Conv(c3 + 2 * c4, c2, 1, 1)

    def forward(self, x):
        y = list(self.cv1(x).chunk(2, 1))
        y.extend(m(y[-1]) for m in [self.cv2, self.cv3])
        cat = self.caa(torch.cat(y, 1))
        return self.cv4(cat)

In [40]:
from einops import rearrange

class TSSA(nn.Module):
    """
    Token Statistics Self-Attention — ported faithfully from the official
    ToST implementation (Wu et al., ICLR 2025; the paper's ref [20]).
    Linear-complexity: no N×N similarity matrix. Operates on (B, N, C) tokens.
    """
    def __init__(self, dim, num_heads=8, qkv_bias=False):
        super().__init__()
        self.heads = num_heads
        self.attend = nn.Softmax(dim=1)
        self.qkv = nn.Linear(dim, dim, bias=qkv_bias)
        self.temp = nn.Parameter(torch.ones(num_heads, 1))
        self.to_out = nn.Linear(dim, dim)

    def forward(self, x):                                  # x: (B, N, C)
        w = rearrange(self.qkv(x), 'b n (h d) -> b h n d', h=self.heads)
        w_normed = torch.nn.functional.normalize(w, dim=-2)      # normalize over tokens
        Pi = self.attend(torch.sum(w_normed ** 2, dim=-1) * self.temp)  # (B,h,N)
        dots = torch.matmul(
            (Pi / (Pi.sum(dim=-1, keepdim=True) + 1e-8)).unsqueeze(-2),
            w ** 2)
        attn = 1.0 / (1 + dots)
        out = -torch.mul(w.mul(Pi.unsqueeze(-1)), attn)
        out = rearrange(out, 'b h n d -> b n (h d)')
        return self.to_out(out)

In [41]:
import math

def build_2d_sincos_pos_embed(w, h, dim, temperature=10000.0, device=None, dtype=None):
    """2D sine-cosine positional embedding, RT-DETR/AIFI convention. -> (1, w*h, dim)"""
    assert dim % 4 == 0, "embed dim must be divisible by 4 for 2D sin-cos"
    gx = torch.arange(w, device=device, dtype=torch.float32)
    gy = torch.arange(h, device=device, dtype=torch.float32)
    gx, gy = torch.meshgrid(gx, gy, indexing='ij')
    pos_dim = dim // 4
    omega = torch.arange(pos_dim, device=device, dtype=torch.float32) / pos_dim
    omega = 1.0 / (temperature ** omega)
    ox = gx.flatten()[..., None] @ omega[None]
    oy = gy.flatten()[..., None] @ omega[None]
    pe = torch.cat([ox.sin(), ox.cos(), oy.sin(), oy.cos()], dim=1)[None]
    return pe.to(dtype=dtype) if dtype is not None else pe

In [42]:
class TAIFI(nn.Module):
    """
    Token-Aware Interaction with Feature Integration (Paper §2.4).
    Same wrapper as RT-DETR's AIFI (flatten -> +pos-embed -> token op -> FFN ->
    reshape), but MHSA is replaced by TSSA. Applied to the deepest map (S5).
    Post-norm Transformer block, matching the paper's description.
    """
    def __init__(self, c, num_heads=8, ffn_ratio=4, dropout=0.0, act=nn.GELU):
        super().__init__()
        self.tssa = TSSA(c, num_heads=num_heads)
        self.ffn = nn.Sequential(
            nn.Linear(c, c * ffn_ratio), act(),
            nn.Dropout(dropout), nn.Linear(c * ffn_ratio, c))
        self.norm1 = nn.LayerNorm(c)
        self.norm2 = nn.LayerNorm(c)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):                                  # x: (B, C, H, W)
        b, c, h, w = x.shape
        pos = build_2d_sincos_pos_embed(w, h, c, device=x.device, dtype=x.dtype)
        src = x.flatten(2).permute(0, 2, 1)                # (B, HW, C)
        # post-norm: sublayer -> residual -> norm
        src = self.norm1(src + self.drop(self.tssa(src + pos)))
        src = self.norm2(src + self.drop(self.ffn(src)))
        return src.permute(0, 2, 1).view(b, c, h, w).contiguous()

In [43]:
torch.manual_seed(0)

# TSSA on raw token sequence
tssa = TSSA(384, num_heads=8)
tok = torch.randn(2, 400, 384)                     # (B, N=20*20, C)
print("TSSA (B,N,C)    :", tuple(tssa(tok).shape))

# TAIFI on the S5 feature map (20x20, 384ch) — the real use
taifi = TAIFI(384, num_heads=8)
s5 = torch.randn(2, 384, 20, 20)
out = taifi(s5)
print("TAIFI S5->S5    :", tuple(out.shape))

assert tssa(tok).shape == (2, 400, 384)
assert out.shape == (2, 384, 20, 20), "TAIFI must preserve the feature-map shape"

# linear-complexity sanity: no N×N tensor should ever be built.
# quick check that it runs on a longer sequence without exploding memory
big = torch.randn(1, 384, 40, 40)                  # 1600 tokens
assert taifi(big).shape == (1, 384, 40, 40)

# gradient sanity
out.mean().backward()
assert any(p.grad is not None for p in taifi.parameters())
print("\nOK — TSSA and TAIFI produce correct shapes, handle long sequences, and are differentiable.")

TSSA (B,N,C)    : (2, 400, 384)
TAIFI S5->S5    : (2, 384, 20, 20)

OK — TSSA and TAIFI produce correct shapes, handle long sequences, and are differentiable.


In [44]:
!pip install -q ultralytics einops
import ultralytics
print("ultralytics", ultralytics.__version__)

ultralytics 8.4.96


In [ ]:
import torch.nn as nn
import ultralytics.nn.tasks as tasks
import ultralytics.nn.modules as ult_modules

_CUSTOM = {
    "GMONet": GMONet, "GMOBlock": GMOBlock, "DMambaOut": DMambaOut,
    "GatedCNNBlock": GatedCNNBlock, "LayerNorm2d": LayerNorm2d,
    "GSConv": GSConv, "CAFF": CAFF, "CAA": CAA,
    "RepNCSP": RepNCSP, "RepConv": RepConv, "RepNBottleneck": RepNBottleneck,
    "TAIFI": TAIFI, "TSSA": TSSA,
}

def _register(mapping):
    for name, cls in mapping.items():
        setattr(ult_modules, name, cls)          # resolves via ultralytics.nn.modules
        setattr(tasks, name, cls)                # resolves via parse_model globals()
        for attr in ("base_modules", "global_modules"):
            s = getattr(tasks, attr, None)
            if isinstance(s, (set, frozenset)) and cls not in s:
                setattr(tasks, attr, frozenset(s | {cls}))

_register(_CUSTOM)
print("registered:", list(_CUSTOM))

registered: ['GMONet', 'GMOBlock', 'DMambaOut', 'GatedCNNBlock', 'LayerNorm2d', 'GSConv', 'CAFF', 'CAA', 'RepNCSP', 'RepConv', 'RepNBottleneck', 'TAIFI', 'TSSA']


In [ ]:
# CELL A

import re
import ultralytics.nn.tasks as tasks

with open(tasks.__file__) as f:
    lines = f.read().splitlines(keepends=True)

# extract the module-level parse_model function block
start = next(i for i, l in enumerate(lines) if l.startswith("def parse_model"))
end = start + 1
while end < len(lines):
    l = lines[end]
    if l[:1].strip() and (l.startswith(("def ", "class ", "@"))):  # next top-level def/class
        break
    end += 1
func_src = "".join(lines[start:end])

# inject CAFF, GSConv into the base_modules frozenset
m = re.search(r"base_modules\s*=\s*frozenset\(\s*\{\s*\n", func_src)
assert m, "frozenset opening not found in source"
inject = m.group(0) + "            CAFF,\n            GSConv,\n"
patched = func_src[:m.start()] + inject + func_src[m.end():]


tasks.CAFF = CAFF
tasks.GSConv = GSConv
exec(patched, tasks.__dict__)
print("parse_model re-patched from disk — base_modules now includes CAFF, GSConv")

parse_model re-patched from disk — base_modules now includes CAFF, GSConv


In [ ]:

from ultralytics.nn.modules import Index          
print("using built-in Index:", Index)

class GMONetBackbone(GMONet):
    """GMONet returning [S3, S4, S5] so one YAML layer emits all taps."""
    def forward(self, x):
        s3, s4, s5 = super().forward(x)
        return [s3, s4, s5]

_register({"GMONetBackbone": GMONetBackbone})
print("ok")

using built-in Index: <class 'ultralytics.nn.modules.conv.Index'>
ok


In [48]:
gmo_detr_4a_yaml = """
nc: 12
scales:
  s: [1.00, 1.00, 2048]

backbone:
  - [-1, 1, GMONetBackbone, []]      # 0  -> [S3(256), S4(384), S5(384)]
  - [0, 1, Index, [256, 0]]          # 1  P3/8   256ch
  - [0, 1, Index, [384, 1]]          # 2  P4/16  384ch
  - [0, 1, Index, [384, 2]]          # 3  P5/32  384ch

head:
  - [3, 1, Conv, [256, 1, 1, None, 1, 1, False]]  # 4 input_proj.2 (keep standard)
  - [-1, 1, TAIFI, [256, 8]]                        # 5 TAIFI
  - [-1, 1, GSConv, [256, 1, 1]]                    # 6 Y5  (GSConv)

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]     # 7
  - [2, 1, Conv, [256, 1, 1, None, 1, 1, False]]   # 8 input_proj.1 (keep standard)
  - [[-2, -1], 1, Concat, [1]]                      # 9
  - [-1, 1, CAFF, [256]]                            # 10 F4
  - [-1, 1, GSConv, [256, 1, 1]]                    # 11 Y4  (GSConv)

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]     # 12
  - [1, 1, Conv, [256, 1, 1, None, 1, 1, False]]   # 13 input_proj.0 (keep standard)
  - [[-2, -1], 1, Concat, [1]]                      # 14
  - [-1, 1, CAFF, [256]]                            # 15 X3 (P3 out)

  - [-1, 1, GSConv, [256, 3, 2]]                    # 16 (GSConv, downsample)
  - [[-1, 11], 1, Concat, [1]]                      # 17
  - [-1, 1, CAFF, [256]]                            # 18 (P4 out)

  - [-1, 1, GSConv, [256, 3, 2]]                    # 19 (GSConv, downsample)
  - [[-1, 6], 1, Concat, [1]]                       # 20
  - [-1, 1, CAFF, [256]]                            # 21 (P5 out)

  - [[15, 18, 21], 1, RTDETRDecoder, [nc]]          # 22
"""
with open("gmo_detr_4a.yaml", "w") as f:
    f.write(gmo_detr_4a_yaml)
print("wrote gmo_detr_4a.yaml")

wrote gmo_detr_4a.yaml


In [49]:
# CELL 20
from ultralytics import RTDETR
import torch

model = RTDETR("gmo_detr_4a.yaml")
det = model.model
det.nc = 12
det.model[-1].nc = 12                      # set on the RTDETRDecoder head too
det.names = {i: str(i) for i in range(12)}
det.train()

# ---- probe: confirm each Index selects the intended tap (right data, not just right channels) ----
bb = det.model[0]                         # GMONetBackbone
with torch.no_grad():
    outs = bb(torch.randn(1, 3, 640, 640))
print("backbone taps  :", [tuple(o.shape[1:]) for o in outs])
# expect: [(256, 80, 80), (384, 40, 40), (384, 20, 20)]

# ---- param count ----
n_params = sum(p.numel() for p in det.parameters())
print("total params   :", f"{n_params/1e6:.2f} M")

# ---- forward first: proves the full graph runs end to end ----
imgs = torch.randn(2, 3, 640, 640)
with torch.no_grad():
    _ = det.predict(imgs)
print("forward OK — decoder produced outputs")

# ---- loss + backward with a correctly-shaped batch ----
batch = {
    "img": imgs,
    "cls": torch.tensor([0, 3], dtype=torch.float32).view(-1, 1),
    "bboxes": torch.tensor([[0.5, 0.5, 0.2, 0.2],
                            [0.4, 0.4, 0.1, 0.3]], dtype=torch.float32),
    "batch_idx": torch.tensor([0, 1], dtype=torch.float32),
}
loss, loss_items = det.loss(batch)
loss.sum().backward()
print("loss items     :", loss_items)
print("\nOK — full GMONet+RT-DETR model runs forward, computes loss, and backprops.")

WARNING ⚠️ no model scale passed. Assuming scale='s'.
backbone taps  : [(256, 80, 80), (384, 40, 40), (384, 20, 20)]
total params   : 17.92 M
forward OK — decoder produced outputs
loss items     : tensor([ 2.0737, 95.6192,  1.3125])

OK — full GMONet+RT-DETR model runs forward, computes loss, and backprops.


In [ ]:
import torch

det.eval()
with torch.no_grad():
    _ = det.predict(torch.randn(1, 3, 640, 640))

# now count for real
total = sum(p.numel() for p in det.parameters())
print(f"TOTAL params (post-build) : {total/1e6:.2f} M   (paper: 12.05 M)")

# per-section breakdown so we can see where any gap is
from collections import defaultdict
groups = defaultdict(int)
for name, p in det.named_parameters():
    layer_idx = int(name.split('.')[1]) if name.split('.')[1].isdigit() else -1
    if layer_idx == 0:
        groups['backbone (GMONet)'] += p.numel()
    elif layer_idx == 22:
        groups['decoder (RTDETRDecoder)'] += p.numel()
    else:
        groups['neck (TAIFI+CAFF+GSConv)'] += p.numel()
for k, v in groups.items():
    print(f"  {k:28s}: {v/1e6:5.2f} M")

TOTAL params (post-build) : 17.92 M   (paper: 12.05 M)
  backbone (GMONet)           :  5.33 M
  neck (TAIFI+CAFF+GSConv)    :  5.26 M
  decoder (RTDETRDecoder)     :  7.33 M


In [51]:
import torch

# Overfit a tiny fixed batch — loss MUST drop sharply if the model can learn.
device = "cuda" if torch.cuda.is_available() else "cpu"
det = det.to(device).train()

torch.manual_seed(0)
imgs = torch.randn(2, 3, 640, 640, device=device)
batch = {
    "img": imgs,
    "cls": torch.tensor([[0.], [3.]], device=device),
    "bboxes": torch.tensor([[0.5, 0.5, 0.2, 0.2],
                            [0.4, 0.4, 0.1, 0.3]], device=device),
    "batch_idx": torch.tensor([0., 1.], device=device),
}

opt = torch.optim.AdamW(det.parameters(), lr=1e-4)

# snapshot some CAFF-internal params to prove gradients actually reach them
watch = {n: p.detach().clone() for n, p in det.named_parameters()
         if (".caa." in n or ".cv4." in n)}

print("step |   total loss")
for step in range(60):
    opt.zero_grad()
    loss, items = det.loss(batch)
    loss.sum().backward()
    opt.step()
    if step % 10 == 0 or step == 59:
        print(f"{step:4d} | {loss.sum().item():10.3f}")

now = dict(det.named_parameters())
moved = sum(1 for n, p0 in watch.items() if not torch.equal(p0, now[n]))
print(f"\nCAFF params changed: {moved}/{len(watch)}",
      "-> OK, gradients reach CAFF" if moved == len(watch) else "-> PROBLEM: some frozen")

step |   total loss
   0 |    815.084
  10 |    274.564
  20 |    163.083
  30 |    128.084
  40 |    111.812
  50 |    100.511
  59 |     94.375

CAFF params changed: 52/52 -> OK, gradients reach CAFF


In [ ]:
import os, glob, json

candidates = glob.glob("/kaggle/input/**/*.json", recursive=True)
print("--- JSON files found ---")
for c in candidates:
    print("  ", c)

print("\n--- /kaggle/input tree (top 2 levels) ---")
for root, dirs, files in os.walk("/kaggle/input"):
    depth = root.replace("/kaggle/input", "").count(os.sep)
    if depth <= 2:
        imgs = [f for f in files if f.lower().endswith((".jpg",".jpeg",".png",".bmp"))]
        print("  " * depth, os.path.basename(root) or ".", f"({len(imgs)} imgs, {len(files)} files)")

if candidates:
    with open(candidates[0]) as f:
        coco = json.load(f)
    print("\n--- first JSON:", candidates[0], "---")
    print("keys:", list(coco.keys()))
    print("images:", len(coco.get("images", [])), "| annotations:",
          len(coco.get("annotations", [])), "| categories:", len(coco.get("categories", [])))
    print("sample image record:", coco["images"][0] if coco.get("images") else "none")
    print("sample annotation  :", coco["annotations"][0] if coco.get("annotations") else "none")
    print("categories:", [(c["id"], c["name"]) for c in coco.get("categories", [])])

--- JSON files found ---
   /kaggle/input/datasets/vidishagar/pcb-large-coco/annotations/instances_default.json

--- /kaggle/input tree (top 2 levels) ---
 input (0 imgs, 0 files)
   datasets (0 imgs, 0 files)
     vidishagar (0 imgs, 0 files)

--- first JSON: /kaggle/input/datasets/vidishagar/pcb-large-coco/annotations/instances_default.json ---
keys: ['licenses', 'info', 'categories', 'images', 'annotations']
images: 551 | annotations: 604 | categories: 12
sample image record: {'id': 1, 'width': 1280, 'height': 720, 'file_name': 'categories/Component Crack/Cheack.jpg', 'license': 0, 'flickr_url': '', 'coco_url': '', 'date_captured': 0}
sample annotation  : {'id': 1, 'image_id': 1, 'category_id': 1, 'segmentation': [], 'area': 25886.410000000003, 'bbox': [839.5, 320.1, 139.7, 185.3], 'iscrowd': 0, 'attributes': {'occluded': False, 'rotation': 0.0}}
categories: [(1, 'Component Crack'), (2, 'Component Damage'), (3, 'Component Liftup'), (4, 'Component Missing'), (5, 'Component No Solder'

In [ ]:
import glob, os

imgs = []
for ext in ("*.jpg","*.jpeg","*.png","*.bmp","*.JPG","*.JPEG","*.PNG"):
    imgs += glob.glob(f"/kaggle/input/**/{ext}", recursive=True)
print(f"total images found under /kaggle/input: {len(imgs)}")
for p in imgs[:8]:
    print("  ", p)

cat_dirs = glob.glob("/kaggle/input/**/categories", recursive=True)
print("\n'categories' dirs found:", cat_dirs)

root = "/kaggle/input/datasets/vidishagar/pcb-large-coco"
print(f"\n--- full tree under {root} ---")
for r, d, f in os.walk(root):
    print("  ", r.replace(root, "") or "/", f"({len(f)} files)")

total images found under /kaggle/input: 1746
   /kaggle/input/datasets/vidishagar/pcb-large-dataset/categories/Component Missing/J6 connector missing.jpg
   /kaggle/input/datasets/vidishagar/pcb-large-dataset/categories/Component Missing/F1 MISSING.jpg
   /kaggle/input/datasets/vidishagar/pcb-large-dataset/categories/Component Missing/J10 Missing.jpg
   /kaggle/input/datasets/vidishagar/pcb-large-dataset/categories/Component Missing/L3 Missing.jpg
   /kaggle/input/datasets/vidishagar/pcb-large-dataset/categories/Component Missing/Missing (2).jpg
   /kaggle/input/datasets/vidishagar/pcb-large-dataset/categories/Component Missing/J2 connector missing.jpg
   /kaggle/input/datasets/vidishagar/pcb-large-dataset/categories/Component Missing/D14 missing.jpg
   /kaggle/input/datasets/vidishagar/pcb-large-dataset/categories/Component Missing/D13 LED missing.jpg

'categories' dirs found: ['/kaggle/input/datasets/vidishagar/pcb-large-dataset/categories']

--- full tree under /kaggle/input/dataset

In [54]:
COCO_JSONS = [
    "/kaggle/input/datasets/vidishagar/pcb-large-coco/annotations/instances_default.json",
]
IMAGE_ROOTS = [
    "/kaggle/input/datasets/vidishagar/pcb-large-dataset/categories",
]
NEG_IMAGE_DIR = "/kaggle/input/datasets/vidishagar/non-def-1200"
OUT = "/kaggle/working/dataset"
VAL_FRACTION = 0.2
SEED = 42
print("config set")

config set


In [55]:
import json, os, glob

def load_and_merge(paths):
    images, anns, cats = [], [], {}
    next_img_id, next_ann_id = 1, 1
    cat_name_to_id = {}      # normalized name -> new contiguous 0-indexed id
    for p in paths:
        with open(p) as f: c = json.load(f)
        local_cat = {}
        for cc in c["categories"]:
            norm = cc["name"].strip().lower().replace(" ", "_")
            if norm not in cat_name_to_id:
                cat_name_to_id[norm] = len(cat_name_to_id)
            local_cat[cc["id"]] = cat_name_to_id[norm]
        old_to_new_img = {}
        for im in c["images"]:
            nid = next_img_id; next_img_id += 1
            old_to_new_img[im["id"]] = nid
            images.append({"id": nid, "file_name": im["file_name"],
                           "width": im["width"], "height": im["height"]})
        for a in c["annotations"]:
            anns.append({"id": next_ann_id,
                         "image_id": old_to_new_img[a["image_id"]],
                         "category_id": local_cat[a["category_id"]],
                         "bbox": a["bbox"]})       # [x,y,w,h] absolute pixels
            next_ann_id += 1
    return images, anns, cat_name_to_id

images, anns, cat_map = load_and_merge(COCO_JSONS)
id_to_name = {v: k for k, v in cat_map.items()}
CLASS_NAMES = [id_to_name[i] for i in range(len(id_to_name))]
print(f"merged: {len(images)} images, {len(anns)} annotations, {len(CLASS_NAMES)} classes")
print("classes (YOLO id order):", CLASS_NAMES)

merged: 551 images, 604 annotations, 12 classes
classes (YOLO id order): ['component_crack', 'component_damage', 'component_liftup', 'component_missing', 'component_no_solder', 'component_solder_dry', 'led_damage', 'polarity_wrong', 'ryb_wrong_sequence', 'solder_ball', 'solder_short', 'tombstone']


In [56]:
import os, glob

def index_disk_images(roots):
    disk = []
    for r in roots:
        for ext in ("*.jpg","*.jpeg","*.png","*.bmp","*.JPG","*.JPEG","*.PNG"):
            disk += glob.glob(os.path.join(r, "**", ext), recursive=True)
    return disk

disk_imgs = index_disk_images(IMAGE_ROOTS)
print(f"{len(disk_imgs)} image files on disk under IMAGE_ROOTS")

def resolve(file_name):
    fn = file_name.replace("\\", "/")
    # 1) exact suffix match (handles 'categories/<Class>/<file>' paths)
    for d in disk_imgs:
        if d.replace("\\", "/").endswith(fn):
            return d
    # 2) suffix match after stripping a leading 'categories/' if roots already point inside it
    fn2 = fn.split("categories/", 1)[-1]
    for d in disk_imgs:
        if d.replace("\\", "/").endswith("/" + fn2):
            return d
    # 3) fallback: unique basename
    base = os.path.basename(fn)
    hits = [d for d in disk_imgs if os.path.basename(d) == base]
    return hits[0] if len(hits) == 1 else None

missing = 0
for im in images:
    im["path"] = resolve(im["file_name"])
    if im["path"] is None:
        missing += 1
print(f"resolved {len(images)-missing}/{len(images)} images; {missing} unresolved")
if missing:
    print("examples unresolved:",
          [im["file_name"] for im in images if im["path"] is None][:5])

551 image files on disk under IMAGE_ROOTS
resolved 548/551 images; 3 unresolved
examples unresolved: ['categories/Component No Solder/RED & GROUND wire no solder.jpg', 'categories/Component Solder Dry/j2 DRY SOLDER & SOLDER SHORT.jpg', 'categories/Solder Ball/Q6 &R8 Solder Boll.jpg']


In [57]:
import os, re, glob

def _norm(s):
    # lowercase, drop extension, remove spaces/&/underscores/punctuation for fuzzy compare
    s = os.path.splitext(os.path.basename(s.replace("\\", "/")))[0].lower()
    return re.sub(r'[^a-z0-9]', '', s)

# build normalized index of disk images, keyed WITHIN their parent folder (class dir)
disk_norm = {}
for d in disk_imgs:
    parent = os.path.basename(os.path.dirname(d)).lower()
    disk_norm.setdefault(parent, []).append((_norm(d), d))

fixed = 0
for im in images:
    if im["path"] is not None:
        continue
    fn = im["file_name"].replace("\\", "/")
    cls_dir = fn.split("/")[-2].lower() if "/" in fn else ""
    target = _norm(fn)
    cands = disk_norm.get(cls_dir, [])
    hits = [d for n, d in cands if n == target]
    if len(hits) == 1:
        im["path"] = hits[0]; fixed += 1
    else:
        # last resort: substring match on normalized name within the class dir
        hits = [d for n, d in cands if target in n or n in target]
        if len(hits) == 1:
            im["path"] = hits[0]; fixed += 1

still = sum(1 for im in images if im["path"] is None)
print(f"fuzzy-resolved {fixed} more; {still} still unresolved")
if still:
    print("remaining:", [im["file_name"] for im in images if im["path"] is None])

fuzzy-resolved 3 more; 0 still unresolved


In [58]:
MAX_NEGATIVES = 120

In [59]:
import os, shutil, random, collections, glob, yaml
random.seed(SEED)

by_img = collections.defaultdict(list)
for a in anns: by_img[a["image_id"]].append(a)

imgs_with_path = [im for im in images if im["path"]]
class_freq = collections.Counter(a["category_id"] for a in anns)
def rarest(im):
    cs = [a["category_id"] for a in by_img[im["id"]]]
    return min((class_freq[c] for c in cs), default=10**9)
imgs_sorted = sorted(imgs_with_path, key=rarest)   # rare-class images first

val_target = int(len(imgs_sorted) * VAL_FRACTION)
val_ids, seen_val = set(), collections.Counter()
for im in imgs_sorted:                              # guarantee each class in val
    cs = {a["category_id"] for a in by_img[im["id"]]}
    if any(seen_val[c] == 0 for c in cs) and len(val_ids) < val_target:
        val_ids.add(im["id"])
        for c in cs: seen_val[c] += 1
remaining = [im for im in imgs_sorted if im["id"] not in val_ids]
random.shuffle(remaining)
for im in remaining[: max(0, val_target - len(val_ids))]:
    val_ids.add(im["id"])

def write_split(split, im_list):
    idir = f"{OUT}/images/{split}"; ldir = f"{OUT}/labels/{split}"
    os.makedirs(idir, exist_ok=True); os.makedirs(ldir, exist_ok=True)
    n_box = 0
    for im in im_list:
        stem = f"{im['id']:06d}"
        ext = os.path.splitext(im["path"])[1]
        shutil.copy(im["path"], f"{idir}/{stem}{ext}")
        W, H = im["width"], im["height"]
        lines = []
        for a in by_img[im["id"]]:
            x, y, w, h = a["bbox"]
            cx, cy, nw, nh = (x + w/2)/W, (y + h/2)/H, w/W, h/H
            if nw < 0.001 or nh < 0.001: continue
            lines.append(f"{a['category_id']} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
            n_box += 1
        open(f"{ldir}/{stem}.txt", "w").write("\n".join(lines))
    return n_box

train_imgs = [im for im in imgs_with_path if im["id"] not in val_ids]
val_imgs   = [im for im in imgs_with_path if im["id"] in val_ids]
nb_tr = write_split("train", train_imgs)
nb_va = write_split("val", val_imgs)
print(f"train: {len(train_imgs)} imgs / {nb_tr} boxes | val: {len(val_imgs)} imgs / {nb_va} boxes")

# per-class val coverage (important given the imbalance)
val_cls = collections.Counter()
for im in val_imgs:
    for a in by_img[im["id"]]: val_cls[a["category_id"]] += 1
print("val per-class boxes:", {CLASS_NAMES[k]: v for k, v in sorted(val_cls.items())})

# defect-free negatives -> empty label files in train
if NEG_IMAGE_DIR and os.path.isdir(NEG_IMAGE_DIR):
    negs = []
    for ext in ("*.jpg","*.jpeg","*.png","*.bmp","*.JPG","*.PNG"):
        negs += glob.glob(os.path.join(NEG_IMAGE_DIR, "**", ext), recursive=True)
    for i, np_ in enumerate(negs):
        stem = f"neg_{i:06d}"; ext = os.path.splitext(np_)[1]
        shutil.copy(np_, f"{OUT}/images/train/{stem}{ext}")
        open(f"{OUT}/labels/train/{stem}.txt", "w").write("")
    print(f"added {len(negs)} defect-free negatives to train")

data_yaml = {"path": OUT, "train": "images/train", "val": "images/val",
             "nc": len(CLASS_NAMES), "names": CLASS_NAMES}
os.makedirs(OUT, exist_ok=True)
with open(f"{OUT}/data.yaml", "w") as f: yaml.safe_dump(data_yaml, f, sort_keys=False)
print("\nwrote", f"{OUT}/data.yaml\n")
print(yaml.safe_dump(data_yaml, sort_keys=False))

train: 441 imgs / 484 boxes | val: 110 imgs / 120 boxes
val per-class boxes: {'component_crack': 3, 'component_damage': 2, 'component_liftup': 5, 'component_missing': 9, 'component_no_solder': 26, 'component_solder_dry': 22, 'led_damage': 2, 'polarity_wrong': 11, 'ryb_wrong_sequence': 4, 'solder_ball': 9, 'solder_short': 25, 'tombstone': 2}
added 1195 defect-free negatives to train

wrote /kaggle/working/dataset/data.yaml

path: /kaggle/working/dataset
train: images/train
val: images/val
nc: 12
names:
- component_crack
- component_damage
- component_liftup
- component_missing
- component_no_solder
- component_solder_dry
- led_damage
- polarity_wrong
- ryb_wrong_sequence
- solder_ball
- solder_short
- tombstone



In [ ]:
import os, glob, random
random.seed(SEED)

for p in glob.glob(f"{OUT}/images/train/neg_*") + glob.glob(f"{OUT}/labels/train/neg_*"):
    os.remove(p)
print("cleared old negatives from train")

MAX_NEGATIVES = 120
negs = []
for ext in ("*.jpg","*.jpeg","*.png","*.bmp","*.JPG","*.PNG"):
    negs += glob.glob(os.path.join(NEG_IMAGE_DIR, "**", ext), recursive=True)
random.shuffle(negs)
negs = negs[:MAX_NEGATIVES]

import shutil
for i, np_ in enumerate(negs):
    stem = f"neg_{i:06d}"; ext = os.path.splitext(np_)[1]
    shutil.copy(np_, f"{OUT}/images/train/{stem}{ext}")
    open(f"{OUT}/labels/train/{stem}.txt", "w").write("")   # empty label = negative

# 3) confirm final train composition
n_pos = len(glob.glob(f"{OUT}/images/train/[0-9]*"))
n_neg = len(glob.glob(f"{OUT}/images/train/neg_*"))
print(f"train now: {n_pos} defect imgs + {n_neg} negatives = {n_pos + n_neg} total")
print(f"negative ratio: {n_neg/(n_pos+n_neg):.0%}")

cleared old negatives from train
train now: 441 defect imgs + 120 negatives = 561 total
negative ratio: 21%


In [62]:
import torch

# save the assembled 12-class architecture to a yaml the trainer can reload
# (model.train needs to build from cfg; we pass our yaml + set nc)
device = 0 if torch.cuda.is_available() else "cpu"

results = model.train(
    data=f"{OUT}/data.yaml",
    epochs=200,                 # first real run; bump to 200 (paper) once we see it learn
    imgsz=640,                  # paper setting
    batch=8,                    # paper setting
    optimizer="AdamW",          # paper setting
    lr0=1e-4,                   # paper setting
    weight_decay=1e-4,          # paper setting
    warmup_epochs=3,
    patience=30,                # early-stop if val mAP plateaus
    device=device,
    project="/kaggle/working/runs",
    name="gmo_detr_pcb",
    exist_ok=True,
    seed=42,
    val=True,
    plots=True,
)
print("training done — best weights:", results.save_dir)

Ultralytics 8.4.96 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=gmo_detr_4a.yaml, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=gmo_detr_pcb, nbs=64, nms=False, opset=None, optimize=False, optimizer=Ad

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      1/200      6.87G       1.77      10.17      1.503          1        640: 100% ━━━━━━━━━━━━ 71/71 1.3it/s 55.3s0.8ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 1.1it/s 6.3s0.4ss
                   all        110        120          0          0          0          0

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      2/200      6.89G      1.719     0.4113      1.466         15        640: 0% ──────────── 0/71  0.6s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      2/200      6.98G      1.583     0.5287      1.296          1        640: 100% ━━━━━━━━━━━━ 71/71 1.6it/s 44.5s0.5ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 2.8it/s 2.5s0.4s
                   all        110        120   1.01e-05     0.0152   4.02e-06   5.91e-07

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      3/200      6.97G      1.537     0.4392      1.183         17        640: 0% ──────────── 0/71  0.6s

...

    198/200      7.08G     0.3224     0.6589     0.2144          1        640: 100% ━━━━━━━━━━━━ 71/71 1.6it/s 43.5s0.5ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 2.8it/s 2.5s0.4s
                   all        110        120      0.689      0.412      0.512      0.336

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
    199/200       6.9G      0.248     0.5473     0.2512         10        640: 0% ──────────── 0/71  0.6s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


    199/200      6.93G     0.3042     0.6178     0.1916          1        640: 100% ━━━━━━━━━━━━ 71/71 1.6it/s 43.2s0.4ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 2.9it/s 2.4s0.4s
                   all        110        120      0.717      0.412      0.518      0.345

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
    200/200       6.9G     0.2237     0.4389     0.1115          6        640: 0% ──────────── 0/71  0.6s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


    200/200      6.99G     0.3157     0.6177     0.2067          1        640: 100% ━━━━━━━━━━━━ 71/71 1.6it/s 43.4s0.5ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 2.9it/s 2.5s0.4s
                   all        110        120      0.642      0.455      0.491      0.327

200 epochs completed in 2.634 hours.
Optimizer stripped from /kaggle/working/runs/gmo_detr_pcb/weights/last.pt, 36.4MB
Optimizer stripped from /kaggle/working/runs/gmo_detr_pcb/weights/best.pt, 36.4MB

Validating /kaggle/working/runs/gmo_detr_pcb/weights/best.pt...
Ultralytics 8.4.96 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
gmo_detr_4a summary: 390 layers, 17,886,198 parameters, 0 gradients, 56.0 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 2.7it/s 2.6s0.4s
                   all        110        120      0.682      0.408      0.534       0.35
       comp